In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score

# ============================================================
# LOAD DATA
# ============================================================
DATA_PATH = '/home/georgetsolos/ml-zoomcamp/cohorts/2026/data/course_lead_scoring.csv'
df = pd.read_csv(DATA_PATH)

print("shape:", df.shape)
print("columns:", df.columns.tolist())
print(df['converted'].value_counts())

# ============================================================
# DATA PREPARATION
# ============================================================
# Fill missing: 'NA' for categorical, 0.0 for numerical
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('NA')
    else:
        df[col] = df[col].fillna(0.0)

# Split exactly as specified
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train      = df_train.reset_index(drop=True)
df_val        = df_val.reset_index(drop=True)
df_full_train = df_full_train.reset_index(drop=True)

y_train = df_train['converted'].values
y_val   = df_val['converted'].values

# IMPORTANT: keep 'converted' in df_full_train — KFold needs it.
# Only drop it from the two smaller frames used in Q1–Q4.
del df_train['converted']
del df_val['converted']

# ============================================================
# QUESTION 1 — ROC AUC feature importance
# ============================================================
print("\n========== Q1 ==========")
numerical_cols = ['lead_score', 'number_of_courses_viewed',
                  'interaction_count', 'annual_income']

for col in numerical_cols:
    auc = roc_auc_score(y_train, df_train[col])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train[col])
    print(f"{col:30s}: AUC = {auc:.3f}")

# ============================================================
# QUESTION 2 — Logistic regression, validation AUC
# ============================================================
print("\n========== Q2 ==========")
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val   = dv.transform(df_val.to_dict(orient='records'))

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
print(f"Validation AUC = {roc_auc_score(y_val, y_pred):.3f}")

# ============================================================
# QUESTION 3 — Precision/Recall intersection threshold
# ============================================================
print("\n========== Q3 ==========")
thresholds = np.arange(0.0, 1.01, 0.01)

best_t_pr, min_diff = None, float('inf')
for t in thresholds:
    pred = (y_pred >= t).astype(int)
    if pred.sum() == 0:
        continue
    p = precision_score(y_val, pred, zero_division=0)
    r = recall_score(y_val, pred, zero_division=0)
    if p == 0 and r == 0:
        continue
    if abs(p - r) < min_diff:
        min_diff = abs(p - r)
        best_t_pr = t

print(f"Precision/Recall intersection threshold = {best_t_pr:.2f}")

# ============================================================
# QUESTION 4 — Max F1 threshold
# ============================================================
print("\n========== Q4 ==========")
best_f1, best_t_f1 = 0.0, None
for t in thresholds:
    pred = (y_pred >= t).astype(int)
    if pred.sum() == 0:
        continue
    p = precision_score(y_val, pred, zero_division=0)
    r = recall_score(y_val, pred, zero_division=0)
    if p + r == 0:
        continue
    f1 = 2 * p * r / (p + r)
    if f1 > best_f1:
        best_f1, best_t_f1 = f1, t

print(f"Max F1 = {best_f1:.3f} at threshold = {best_t_f1:.2f}")

# ============================================================
# Q5 & Q6 — 5-Fold CV helper
# ============================================================
kfold = KFold(n_splits=5, shuffle=True, random_state=1)

def cv_scores(C):
    """Return array of AUC scores across the 5 folds."""
    scores = []
    for tr_idx, va_idx in kfold.split(df_full_train):
        df_tr = df_full_train.iloc[tr_idx]
        df_va = df_full_train.iloc[va_idx]

        y_tr = df_tr['converted'].values
        y_va = df_va['converted'].values

        df_tr = df_tr.drop(columns=['converted'])
        df_va = df_va.drop(columns=['converted'])

        dv_fold = DictVectorizer(sparse=False)
        X_tr = dv_fold.fit_transform(df_tr.to_dict(orient='records'))
        X_va = dv_fold.transform(df_va.to_dict(orient='records'))

        m = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
        m.fit(X_tr, y_tr)

        scores.append(roc_auc_score(y_va, m.predict_proba(X_va)[:, 1]))
    return np.array(scores)

# ============================================================
# QUESTION 5 — std of scores with C=1.0
# ============================================================
print("\n========== Q5 ==========")
scores_C1 = cv_scores(1.0)
print(f"mean AUC = {scores_C1.mean():.3f}")
print(f"std      = {scores_C1.std():.3f}")
print("scores   =", np.round(scores_C1, 3))

# ============================================================
# QUESTION 6 — Hyperparameter tuning over C
# ============================================================
print("\n========== Q6 ==========")
results = []
for C in [0.000001, 0.001, 1]:
    s = cv_scores(C)
    mean_auc = round(s.mean(), 3)
    std_auc  = round(s.std(), 3)
    results.append((C, mean_auc, std_auc))
    print(f"C={C:<10}  mean={mean_auc:.3f}  std={std_auc:.3f}")

# Tie-breaking: highest mean -> lowest std -> smallest C
results.sort(key=lambda x: (-x[1], x[2], x[0]))
print(f"\nBest C = {results[0][0]}  (mean={results[0][1]:.3f}, std={results[0][2]:.3f})")

shape: (1462, 9)
columns: ['lead_source', 'industry', 'number_of_courses_viewed', 'annual_income', 'employment_status', 'location', 'interaction_count', 'lead_score', 'converted']
converted
1    905
0    557
Name: count, dtype: int64

========== Q1 ==========
lead_score                    : AUC = 0.614
number_of_courses_viewed      : AUC = 0.764
interaction_count             : AUC = 0.738
annual_income                 : AUC = 0.552

========== Q2 ==========
Validation AUC = 0.817

========== Q3 ==========
Precision/Recall intersection threshold = 0.64

========== Q4 ==========
Max F1 = 0.812 at threshold = 0.57

========== Q5 ==========
mean AUC = 0.822
std      = 0.036
scores   = [0.806 0.871 0.774 0.801 0.855]

========== Q6 ==========
C=1e-06       mean=0.560  std=0.024
C=0.001       mean=0.867  std=0.029
C=1           mean=0.822  std=0.036

Best C = 0.001  (mean=0.867, std=0.029)
